# Data Processing for GHG and Trade Networks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# === GHG Processing ===
def process_ghg_flows(ghg_path, years):
    """Reads OECD GHG CSV and builds a dictionary of emissions matrices (NxN) by year."""
    ghgdf = pd.read_csv(ghg_path)
    df_co2 = ghgdf[['TIME_PERIOD', 'EXPORTER', 'IMPORTER', 'OBS_VALUE']].copy()

    code_fix_map = {'WXD': 'ROW'}  # unify "Rest of World" code
    df_co2['EXPORTER'] = df_co2['EXPORTER'].replace(code_fix_map)
    df_co2['IMPORTER'] = df_co2['IMPORTER'].replace(code_fix_map)

    ghg_flows_dict = {}
    for year in years:
        yearly = df_co2[df_co2['TIME_PERIOD'] == year]
        pivot = yearly.pivot_table(index='EXPORTER', columns='IMPORTER', values='OBS_VALUE', aggfunc='sum').fillna(0)
        ghg_flows_dict[year] = pivot

    return ghg_flows_dict


# === Trade Processing ===
def add_country_code_levels(df):
    """Splits country codes into a MultiIndex: (country_code, label)."""
    row_iso = df.index.to_series().str[:3]
    col_iso = pd.Series(df.columns, index=df.columns).str[:3]

    df.index = pd.MultiIndex.from_arrays([row_iso, df.index.str[4:]], names=['country_code', 'label'])
    df.columns = pd.MultiIndex.from_arrays([col_iso, df.columns.str[4:]], names=['country_code', 'label'])

    return df


def process_trade_flows(trade_file_template, years):
    """Processes trade flow CSVs for multiple years and returns a dictionary of NxN DataFrames."""
    trade_flows_dict = {}

    for year in years:
        try:
            file_path = trade_file_template.format(year=year)
            df = pd.read_csv(file_path, index_col=0)

            df = add_country_code_levels(df)
            df = df.drop(df.index[-3:])                # Drop last 3 rows
            df = df.drop(df.columns[-1], axis=1)       # Drop last column

            df_agg = df.groupby(level=0, axis=0).sum().groupby(level=0, axis=1).sum()
            trade_flows_dict[year] = df_agg
        except FileNotFoundError:
            print(f"File not found for year {year}, skipping...")

    return trade_flows_dict

def process_trade_flows_wsectors(trade_file_template, years, sectors_of_interest):
    """
    Processes trade flow CSVs for multiple years, filters by sector, and returns a dictionary of NxN DataFrames.
    Keeps only flows where both source and target sectors are in `sectors_of_interest`.
    """
    trade_flows_dict = {}

    for year in years:
        try:
            file_path = trade_file_template.format(year=year)
            df = pd.read_csv(file_path, index_col=0)
            df = add_country_code_levels(df)

            # Optional: drop last 3 rows/columns if they are non-data
            df = df.drop(df.index[-3:])
            df = df.drop(df.columns[-1], axis=1)

            # Filter by sector in both rows and columns
            df = df[df.index.get_level_values('label').isin(sectors_of_interest)]
            df = df.loc[:, df.columns.get_level_values('label').isin(sectors_of_interest)]

            # Aggregate to country x country
            df_agg = df.groupby(level=0, axis=0).sum().groupby(level=0, axis=1).sum()

            trade_flows_dict[year] = df_agg
        except FileNotFoundError:
            print(f"File not found for year {year}, skipping...")

    return trade_flows_dict


# === Diagonal-zero version of trade flows ===
def remove_trade_diagonal(trade_dict, years):
    """Sets diagonal values (intra-country trade) to zero."""
    result = {}
    for year in years:
        df = trade_dict[year].copy()
        np.fill_diagonal(df.values, 0)
        result[year] = pd.DataFrame(df, index=df.index, columns=df.columns)
    return result

# === Combine all years ===
def get_ghg_vs_trade_all_years(ghg_dict, trade_dict, years):
    """Returns a long-form DataFrame of GHG vs trade for all years."""
    all_years_data = []

    for year in years:
        df_ghg = ghg_dict.get(year)
        df_trade = trade_dict.get(year)
        if df_ghg is None or df_trade is None:
            continue

        ghg_long = df_ghg.stack().rename('ghg')
        trade_long = df_trade.stack().rename('trade')

        combined = pd.concat([ghg_long, trade_long], axis=1).dropna()
        combined['ghg_share'] = combined['ghg'] / combined['ghg'].sum()
        combined['trade_share'] = combined['trade'] / combined['trade'].sum()
        combined = combined.reset_index()
        combined['year'] = year

        all_years_data.append(combined)

    return pd.concat(all_years_data, ignore_index=True)


# === GDP Merge ===
def merge_gdppc(combined_df, gdp_df):
    """Merges GDP per capita onto combined_df by exporter and importer."""
    gdp_exp = gdp_df.rename(columns={'Code': 'Source', 'Year': 'year', 'gdppc': 'gdppc_exp'})
    merged = pd.merge(combined_df, gdp_exp, on=['Source', 'year'], how='left')

    gdp_imp = gdp_df.rename(columns={'Code': 'Target', 'Year': 'year', 'gdppc': 'gdppc_imp'})
    merged = pd.merge(merged, gdp_imp, on=['Target', 'year'], how='left')

    return merged


# === Execution ===
years = range(1995, 2021)

# Process input data
ghg_flows_dict = process_ghg_flows("Data/Raw/OECD_GHG.csv", years)

# ghg_flows_dict = process_ghg_flows("OECD_GHG_Primary.csv", years)
# ghg_flows_dict = process_ghg_flows("OECD_GHG_Secondary.csv", years)
# ghg_flows_dict = process_ghg_flows("OECD_GHG_Services.csv", years)

primary_sectors = ['A01_02','A03', 'B05_06','B07_08','B09']
# new_primary_sectors = ['A01_02','A03', 'B05_06','B07_08','B09','C19','C20','C21','C22'
#                      ,'C23','C24','C25']        
              
secondary_sectors = ['C10T12','C13T15','C16','C17_18','C19','C20','C21','C22'
                     ,'C23','C24','C25','C26','C27','C28','C29','C30','C31T33','D','E']   
# new_secondary_sectors = ['C10T12','C13T15','C16','C17_18','C26','C27','C28','C29','C30','C31T33','D','E']   
              
services_sectors = ['F', 'G', 'H49', 'H50', 'H51', 'H52', 'H53', 'I', 'J58T60', 'J61', 'J62_63', 'K', 
                    'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T']

trade_flows_dict = process_trade_flows("Data/Raw/{year}_SML.csv", years)
# trade_flows_dict = process_trade_flows_wsectors("{year}_SML.csv", years, sectors_of_interest=primary_sectors)
# trade_flows_dict = process_trade_flows_wsectors("{year}_SML.csv", years, sectors_of_interest=secondary_sectors)
# trade_flows_dict = process_trade_flows_wsectors("{year}_SML.csv", years, sectors_of_interest=service_sectors)

Ex_trade_flows_dict = remove_trade_diagonal(trade_flows_dict, years)

# All years combined
combined_all_years = get_ghg_vs_trade_all_years(ghg_flows_dict, Ex_trade_flows_dict, years)
combined_all_years = combined_all_years.rename(columns={'level_0': 'Source', 'level_1': 'Target'})
# combined_df = plot_ghg_vs_trade_share(ghg_flows_dict[2019], Ex_trade_flows_dict[2019], year=2019)

# Merge with GDP
gdp_df = pd.read_csv('Data/Raw/gdp-per-capita-worldbank.csv')
gdp_df = gdp_df.rename(columns={'GDP per capita, PPP (constant 2021 international $)': 'gdppc'})
gdp_df = gdp_df[['Code', 'Year', 'gdppc']]

merged_df = merge_gdppc(combined_all_years, gdp_df)
merged_df['diff'] = merged_df['trade_share'] - merged_df['ghg_share']
# merged_df.to_csv('merged_df_primary.csv', index=False)
# merged_df.to_csv('merged_df_secondary.csv', index=False)
# merged_df.to_csv('merged_df_services.csv', index=False)
merged_df.head()